# NER 中医药命名体识别

## 加载处理数据

In [ ]:
categories = set()

def load_data(data_file):
    Data = {}
    with open (data_file, "rt", encoding="utf-8") as f:
        # 文本使用空行进行分割句子
        for idx, line in enumerate(f.read().split("\n\n")):
            if not line:
                break
            sentence, labels = "", []
            for i, item in enumerate(line.split("\n")):
                char, tag = item.split(" ")
                sentence += char
                if tag.startswith("B"):
                    labels.append([i, i, char, tag[2:]])   # Remove the B- or I-
                    categories.add(tag[2:])
                elif tag.startswith("I"):
                    labels[-1][1] = i
                    labels[-1][2] += char
            Data[idx] = {
                "sentence" : sentence,
                "labels" : labels
            }
    return Data

In [2]:
path = "dataset/ner_data/medical.train"
path_dev = "dataset/ner_data/medical.dev"

ds_train = load_data(path)
ds_dev = load_data(path_dev)

In [3]:
for idx, example in ds_train.items():
    print(idx, example)
    if idx >= 3:
        break
print("="*50)
for idx, example in ds_dev.items():
    print(idx, example)
    if idx >= 2:
        break

0 {'sentence': '现头昏口苦', 'labels': [[3, 4, '口苦', '临床表现']]}
1 {'sentence': '目的观察复方丁香开胃贴外敷神阙穴治疗慢性心功能不全伴功能性消化不良的临床疗效', 'labels': [[4, 10, '复方丁香开胃贴', '中医治疗'], [20, 32, '心功能不全伴功能性消化不良', '西医诊断']]}
2 {'sentence': '舒肝和胃消痞汤；功能性消化不良', 'labels': [[8, 14, '功能性消化不良', '西医诊断']]}
3 {'sentence': '患者３ａ前咯血，被诊断为肺结核，住院４０余天时出现腹痛，经治疗好转，但时有发作，坚持服抗痨药３ａ后，因腹痛基本缓解，肺结核治愈而停药', 'labels': [[5, 6, '咯血', '临床表现'], [12, 14, '肺结核', '西医诊断'], [58, 60, '肺结核', '西医诊断']]}
0 {'sentence': '投活络效灵丹加味：当归、丹参各１５ｇ，生乳香、生没药各６ｇ，柴胡１２ｇ，白芍、黄芩、大黄各１０ｇ，蒲公英３０ｇ，甘草５ｇ', 'labels': [[1, 5, '活络效灵丹', '方剂'], [9, 10, '当归', '中药'], [12, 13, '丹参', '中药'], [19, 21, '生乳香', '中药'], [23, 25, '生没药', '中药'], [30, 31, '柴胡', '中药'], [39, 40, '黄芩', '中药'], [42, 43, '大黄', '中药'], [49, 51, '蒲公英', '中药'], [56, 57, '甘草', '中药']]}
1 {'sentence': '目的补气健脾升清法治疗糖尿病性功能性消化不良的临床效果', 'labels': [[11, 21, '糖尿病性功能性消化不良', '西医诊断']]}
2 {'sentence': '结论温脾散穴位敷贴联合理中复元方可改善脾虚痰瘀型慢性萎缩性胃炎患者临床症状（尤其是胃窦大弯侧、胃体小弯侧萎缩），值得推广应用', 'labels': [[2, 8, '温脾散穴位敷贴', '中医治疗'], [19, 22, '脾虚痰瘀', '中医证候'], [24, 30, '慢性萎缩性胃炎

In [ ]:
from datasets import Dataset
from utils import convert_to_sft_format

ds_train_json = convert_to_sft_format(ds_train)
ds_valid_json = convert_to_sft_format(ds_dev)

train_tmp = Dataset.from_list(ds_train_json)
valid_tmp = Dataset.from_list(ds_valid_json)
train_tmp, valid_tmp

/root/miniconda3/envs/state3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(Dataset({
     features: ['instruction', 'input', 'output'],
     num_rows: 5259
 }),
 Dataset({
     features: ['instruction', 'input', 'output'],
     num_rows: 657
 }))

In [6]:
from transformers import AutoTokenizer

model_path = "./model/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_path)

In [7]:
def process_func(example):
    max_length = 384

    system_prompt = """你是一个专业的医学信息抽取助手。
    请根据输入句子识别所有的实体，输出 JSON 格式，包含字段：
    text, type。

    示例：
    句子：气滞胃痛颗粒联合乳果糖治疗便秘型肠易激综合征临床研究
    输出：
    {
    "entities": [
        {"text": "气滞胃痛颗粒", "type": "中医治疗"},
        {"text": "便秘型肠易激综合征", "type": "西医诊断"},
    ]
    }
    """

    user_prompt = f"{example['instruction']}\n句子：{example['input']}"
    assistant_resp = example["output"]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_resp},
    ]

    # 获取完整的 tokenized
    tokenized_full = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False, 
        max_length=max_length,
        truncation=True,
        return_tensors=None,
    )

    attention_mask = [1] * len(tokenized_full)
    
    # 获取 system + user 的 tokenized
    tokenized_prefix = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        tokenize=True,
        add_generation_prompt=True, # 添加 <|im_start|>assistant\n 的前缀
        return_tensors=None,
    )

    # Labels: mask 掉 prefix_len 之前的部分
    prefix_len = len(tokenized_prefix)
    labels = [-100] * min(prefix_len, len(tokenized_full)) + tokenized_full[min(prefix_len, len(tokenized_full)):]

    return {
        "input_ids": tokenized_full,
        "attention_mask": attention_mask,
        "labels": labels,
    }

train_data = train_tmp.map(process_func, remove_columns=train_tmp.column_names)
valid_data = valid_tmp.map(process_func, remove_columns=valid_tmp.column_names)

Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 657/657 [00:01<00:00, 634.39 examples/s]


In [8]:
train_tmp[20]

{'instruction': '你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并输出标准 JSON。',
 'input': '按：患者以“腹痛腹泻１０年余”为主诉，属于中医“泄泻”的范畴',
 'output': '{"entities": [{"text": "腹痛", "type": "临床表现"}, {"text": "腹泻", "type": "临床表现"}, {"text": "泄泻", "type": "中医诊断"}]}'}

In [9]:
print(tokenizer.decode(train_data[20]["input_ids"], skip_special_tokens=False))

<|im_start|>system
你是一个专业的医学信息抽取助手。
    请根据输入句子识别所有的实体，输出 JSON 格式，包含字段：
    text, type。

    示例：
    句子：气滞胃痛颗粒联合乳果糖治疗便秘型肠易激综合征临床研究
    输出：
    {
    "entities": [
        {"text": "气滞胃痛颗粒", "type": "中医治疗"},
        {"text": "便秘型肠易激综合征", "type": "西医诊断"},
    ]
    }
    <|im_end|>
<|im_start|>user
你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并输出标准 JSON。
句子：按：患者以“腹痛腹泻１０年余”为主诉，属于中医“泄泻”的范畴<|im_end|>
<|im_start|>assistant
{"entities": [{"text": "腹痛", "type": "临床表现"}, {"text": "腹泻", "type": "临床表现"}, {"text": "泄泻", "type": "中医诊断"}]}<|im_end|>



In [12]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

current_device = torch.cuda.current_device()

model = AutoModelForCausalLM.from_pretrained(model_path, 
                                            #   quantization_config=bnb_config,
                                              device_map="auto",)

config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    inference_mode=False,  # 训练模式
    r=8,  # Lora 秩
    lora_alpha=32,  # Lora alaph，具体作用参见 Lora 原理
    lora_dropout=0.1,  # Dropout 比例
)

model = get_peft_model(model, config)
model.print_trainable_parameters()
model.enable_input_require_grads() 

trainable params: 8,544,256 || all params: 1,552,258,560 || trainable%: 0.5504


In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
import torch

args = TrainingArguments(
    output_dir="./output/Qwen2.5_7B-ner_0",
    per_device_train_batch_size=8,
    eval_accumulation_steps=2,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    logging_steps=10,
    num_train_epochs=2,
    save_steps=200,
    learning_rate=5e-5,
    eval_strategy="steps",
    eval_steps=20,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    gradient_checkpointing=True,
    logging_dir="../tf-logs/qwen2.5_7B_ner_0/rus",
    report_to="tensorboard",
    fp16=True,
    # prediction_loss_only=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=valid_data,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    # compute_metrics=compute_metrics
)   


In [14]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss
20,0.282100,0.236178
40,0.171500,0.142147
60,0.122300,0.109787
80,0.105800,0.103302
100,0.102200,0.096165
120,0.085200,0.091463
140,0.090700,0.087553
160,0.081000,0.086579
180,0.080300,0.081411
200,0.080800,0.080717


TrainOutput(global_step=658, training_loss=0.08575574211255395, metrics={'train_runtime': 1000.5362, 'train_samples_per_second': 10.512, 'train_steps_per_second': 0.658, 'total_flos': 2.482233372211507e+16, 'train_loss': 0.08575574211255395, 'epoch': 2.0})

save

In [ ]:
lora_path='./qwen2.5_7B_lora_0'
trainer.model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)

('./qwen2.5_1.5B_lora_3/tokenizer_config.json',
 './qwen2.5_1.5B_lora_3/special_tokens_map.json',
 './qwen2.5_1.5B_lora_3/chat_template.jinja',
 './qwen2.5_1.5B_lora_3/vocab.json',
 './qwen2.5_1.5B_lora_3/merges.txt',
 './qwen2.5_1.5B_lora_3/added_tokens.json',
 './qwen2.5_1.5B_lora_3/tokenizer.json')

## eval

In [1]:
categories = set()

def load_data(data_file):
    Data = {}
    with open (data_file, "rt", encoding="utf-8") as f:
        # 文本使用空行进行分割句子
        for idx, line in enumerate(f.read().split("\n\n")):
            if not line:
                break
            sentence, labels = "", []
            for i, item in enumerate(line.split("\n")):
                char, tag = item.split(" ")
                sentence += char
                if tag.startswith("B"):
                    labels.append([i, i, char, tag[2:]])   # Remove the B- or I-
                    categories.add(tag[2:])
                elif tag.startswith("I"):
                    labels[-1][1] = i
                    labels[-1][2] += char
            Data[idx] = {
                "sentence" : sentence,
                "labels" : labels
            }
    return Data

path_test = "dataset/ner_data/medical.test"

ds_test = load_data(path_test)

In [2]:
import json

def convert_to_sft_format(Data):
    sft_data = []
    for idx, sample in Data.items():
        sentence = sample["sentence"]
        labels = sample["labels"]

        # 构造output
        output = [
            {"text": text, "type": tag}
            for start, end, text, tag in labels
        ]
        prompt = "你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并输出标准 JSON。"

        sft_data.append({
            "instruction": prompt,
            "input": sentence,
            "output": json.dumps({"entities": output}, ensure_ascii=False)
        })
    return sft_data

In [3]:
from datasets import Dataset

ds_test_json = convert_to_sft_format(ds_test)
test_tmp = Dataset.from_list(ds_test_json)
test_tmp

/root/miniconda3/envs/state3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 658
})

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

lora_path = "qwen2.5_1.5B_lora_3"
model_path = "model/Qwen2.5-1.5B-Instruct"

base_model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="left",)

model = PeftModel.from_pretrained(base_model, lora_path)

In [ ]:
def process_func_valid(example):
    max_length = 384

    system_prompt = """你是一个专业的医学信息抽取助手。
    请根据输入句子识别所有的实体，输出 JSON 格式，包含字段：
    text, type。

    示例：
    句子：气滞胃痛颗粒联合乳果糖治疗便秘型肠易激综合征临床研究
    输出：
    {
    "entities": [
        {"text": "气滞胃痛颗粒", "type": "中医治疗"},
        {"text": "便秘型肠易激综合征", "type": "西医诊断"},
    ]
    }
    """

    user_prompt = f"{example['instruction']}\n句子：{example['input']}"
    assistant_resp = example["output"]

    # 获取 system + user 的 tokenized
    tokenized_prefix = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        tokenize=True,
        add_generation_prompt=True, # 添加 <|im_start|>assistant\n 的前缀
        return_tensors=None,
    )
    attention_mask = [1] * len(tokenized_prefix)

    return {
        "input_ids": tokenized_prefix,
        "attention_mask": attention_mask,
        # "labels": labels,
    }

test_data = test_tmp.map(process_func_valid, remove_columns=test_tmp.column_names)

Map:  18%|██████████████████████▌                                                                                                        | 117/658 [00:00<00:00, 1144.04 examples/s]

Map: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 658/658 [00:00<00:00, 1293.02 examples/s]


In [6]:
res = tokenizer.decode(test_data[6]["input_ids"])
res, len(test_data[6]["input_ids"]), len(test_data[6]["attention_mask"])


('<|im_start|>system\n你是一个专业的医学信息抽取助手。\n    请根据输入句子识别所有的实体，输出 JSON 格式，包含字段：\n    text, type。\n\n    示例：\n    句子：气滞胃痛颗粒联合乳果糖治疗便秘型肠易激综合征临床研究\n    输出：\n    {\n    "entities": [\n        {"text": "气滞胃痛颗粒", "type": "中医治疗"},\n        {"text": "便秘型肠易激综合征", "type": "西医诊断"},\n    ]\n    }\n    <|im_end|>\n<|im_start|>user\n你是一个专业的生物医学信息抽取助手。请从用户提供的句子识别所有的实体，并输出标准 JSON。\n句子：目的：探讨七方胃痛胶囊治疗肝郁脾虚型功能消化不良的效果<|im_end|>\n<|im_start|>assistant\n',
 166,
 166)

In [7]:
test_tmp["output"][3]

'{"entities": [{"text": "腹痛", "type": "临床表现"}]}'

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from tqdm import tqdm
import torch

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

valid_loader = DataLoader(
    test_data,
    batch_size=8,  
    collate_fn=data_collator,
    shuffle=False
)

model.eval()
all_preds = []
refs = test_tmp["output"]

for batch in tqdm(valid_loader):
    with torch.no_grad():
        outputs = model.generate(
            **{k: v.to(model.device) for k, v in batch.items()},
            max_new_tokens=128,
            do_sample=False,
        )
    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    all_preds.extend(preds)


  0%|                                                                                                                                                        | 0/83 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 83/83 [04:46<00:00,  3.45s/it]


In [10]:
preds = []
for item in all_preds:
    if 'assistant\n' in item:
        json_str = item.split('assistant\n', 1)[1]
        preds.append(json_str)

In [24]:
from utils import compute_f1, format_evaluation_results

metrics = compute_f1(preds, refs)

format_evaluation_results(metrics)

🎯 医学命名实体识别评估结果

📊 按实体类型详细结果:
------------------------------------------------------------
实体类型         精确率(P)     召回率(R)     F1分数      
------------------------------------------------------------
中医治则            69.23%    38.30%    49.32%
中医治疗            74.60%    70.68%    72.59%
中医证候            78.48%    78.48%    78.48%
中医诊断            65.12%    58.33%    61.54%
中药              81.05%    47.33%    59.76%
临床表现            73.33%    61.60%    66.96%
其他临床表现           0.00%     0.00%     0.00%
其他治疗            54.55%    50.00%    52.17%
方剂              56.73%    53.15%    54.88%
西医治疗            77.27%    64.15%    70.10%
西医诊断            87.61%    87.88%    87.75%

📈 总体评估指标:
----------------------------------------
🏆 宏平均F1:        59.41%
🎯 微平均精确率:    76.83%
🔍 微平均召回率:    65.93%
⚡ 微平均F1:        70.96%

📋 性能分析:
----------------------------------------
✅ 最佳表现: 西医诊断 (F1: 87.75%)
❌ 最差表现: 其他临床表现 (F1: 0.00%)

📈 F1分数分布:
  优秀 (≥80%): 1 个类型
  良好 (60-80%): 5 个类型
  需改进 (<60%): 5 个类型
